# CDT Distance Field Demo

This notebook demonstrates VoxelCAD's CDT (chamfer distance transform) precision path:
- Multi-isovalue surface extraction from a single distance field
- Offset surface STL export for shell/wall thickness control
- Distance field visualization (volume cross-sections)
- Performance comparison: fast path vs CDT precision path

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import time
import pyvista as pv
from voxelcad import Sphere, Cube, GyroidCube
import voxelcad.environment as ENV
ENV.use_cython = True

## 1. CDT Distance Field as PyVista Volume

The CDT computes signed distances from each voxel to the nearest surface.
Positive = inside, negative = outside. Values are in real units (mm).

In [ ]:
s = Sphere(r=5, voxel_size=0.1)
grid = s.render_cdt_grid(mc_stride=2)

print(f"Grid dimensions: {grid.dimensions}")
print(f"Spacing: {grid.spacing}")
cdt = grid.cell_data['cdt_distance']
print(f"Distance range: [{cdt.min():.2f}, {cdt.max():.2f}] mm")

## 2. Multi-Isovalue Surface Extraction

Extract concentric surfaces at different distances from the boundary.
One CDT computation, many surfaces — the distance field is computed once.

In [ ]:
g = GyroidCube(size=10, voxel_size=0.1)
grid = g.render_cdt_grid(mc_stride=2)

# Extract surfaces at 0mm (boundary), 0.5mm inward, 1.0mm inward
isovalues = [0.0, 0.3, 0.6]
contours = grid.contour(isovalues, scalars='cdt_distance')

print(f"Multi-isovalue contours: {contours.n_points} points, {contours.n_cells} cells")
contours.plot(scalars='cdt_distance', cmap='coolwarm', opacity=0.7)

## 3. Offset Surface STL Export

Use `method='cdt'` with non-zero `isovalue` to export offset surfaces.
This is useful for creating shells, wall thickness control, or toleranced fits.

In [ ]:
import os, tempfile

model = Sphere(r=5, voxel_size=0.1)

with tempfile.TemporaryDirectory() as td:
    # Standard surface (isovalue=0)
    model.export(os.path.join(td, 'boundary.stl'), method='cdt', mc_stride=2)
    
    # Offset inward by 0.5mm
    model.export(os.path.join(td, 'inset_0.5mm.stl'), method='cdt', 
                 mc_stride=2, isovalue=0.5)
    
    # Offset inward by 1.0mm
    model.export(os.path.join(td, 'inset_1.0mm.stl'), method='cdt',
                 mc_stride=2, isovalue=1.0)
    
    for f in sorted(os.listdir(td)):
        sz = os.path.getsize(os.path.join(td, f))
        print(f'{f}: {sz:,} bytes')

## 4. Distance Field Cross-Section

Slice the CDT volume to visualize the distance field as a 2D heatmap.

In [ ]:
import matplotlib.pyplot as plt
from voxelcad._kernels import compute_cdt_field

g = GyroidCube(size=10, voxel_size=0.1)
g.render_volume()
rx, ry, rz = g.grid.res_vector
vsv = g.grid.voxel_size_vector

cdt = compute_cdt_field(g.voxel_data, rx, ry, rz, stride=2, voxel_size=vsv)

# Mid-Z cross-section
mid_z = cdt.shape[2] // 2
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, z_off, title in zip(axes, [-10, 0, 10],
                             ['Z - 10', 'Z center', 'Z + 10']):
    sl = cdt[:, :, mid_z + z_off]
    im = ax.imshow(sl.T, origin='lower', cmap='RdBu_r',
                   vmin=-cdt.max(), vmax=cdt.max())
    ax.contour(sl.T, levels=[0.0], colors='black', linewidths=1.5)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='Distance (mm)')

plt.suptitle('CDT Distance Field Cross-Sections (GyroidCube)', y=1.02)
plt.tight_layout()
plt.show()

## 5. Performance Comparison: Fast vs CDT

The fast path (default) streams directly from packed bits with ~5MB constant memory.
The CDT path materializes the distance field first, then streams through the same backend.

In [ ]:
from voxelcad.utils.spectral import compute_butterworth_kernel
from voxelcad._kernels._fused_parallel import (
    fused_mesh_export, compute_sdf_cdt)

kern = compute_butterworth_kernel(order=4, cutoff=0.25, radius=3)

print(f'{"Res":>5s}  {"Fast (ms)":>10s}  {"CDT (ms)":>10s}  {"Ratio":>6s}  {"SDF mem":>8s}')
print('-' * 50)

for res in [64, 128, 256]:
    g = GyroidCube(size=10, voxel_size=10.0/res)
    g.render_volume()
    rx, ry, rz = g.grid.res_vector
    vsv = g.grid.voxel_size_vector
    stride = 2
    
    # Fast path
    times = []
    for _ in range(5):
        t0 = time.perf_counter()
        fused_mesh_export(g.voxel_data, rx, ry, rz, kern['int8'],
                          vsv[0], vsv[1], vsv[2], stride=stride)
        times.append(time.perf_counter() - t0)
    fast_ms = min(times) * 1000
    
    # CDT path
    times = []
    for _ in range(5):
        t0 = time.perf_counter()
        sdf = compute_sdf_cdt(g.voxel_data, rx, ry, rz, stride)
        fused_mesh_export(g.voxel_data, rx, ry, rz, kern['int8'],
                          vsv[0], vsv[1], vsv[2], stride=stride,
                          sdf_input=sdf)
        times.append(time.perf_counter() - t0)
    cdt_ms = min(times) * 1000
    
    sx = int(np.ceil(rx / stride)) + 2
    sdf_mb = sx**3 / 1e6
    
    print(f'{res:5d}  {fast_ms:10.1f}  {cdt_ms:10.1f}  {cdt_ms/fast_ms:6.2f}x  {sdf_mb:7.1f}M')

## 6. Boolean + Offset Workflow

Combine CSG boolean operations with CDT offset surfaces for toleranced designs.

In [ ]:
# Sphere with cube cutout, then offset the surface inward
sphere = Sphere(r=5, voxel_size=0.1)
cube = Cube(size=7, voxel_size=0.1, center=True)
model = sphere & cube  # intersection

# Render CDT grid to see the distance field
grid = model.render_cdt_grid(mc_stride=2)
cdt = grid.cell_data['cdt_distance']
print(f'Boolean model CDT: range [{cdt.min():.2f}, {cdt.max():.2f}] mm')

# Extract boundary and offset surfaces
boundary = grid.contour([0.0], scalars='cdt_distance')
offset = grid.contour([0.5], scalars='cdt_distance')
print(f'Boundary: {boundary.n_cells} faces')
print(f'Offset 0.5mm: {offset.n_cells} faces')